# 第4回: ReAct

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session04/session04_react.ipynb)

- LLMが外部関数を呼び出す **Tool Use（Function Calling）** と、思考(Thought) → 行動(Action) → 観測(Observation) を繰り返す **ReAct** の基本構造を扱う
- LangChainを使用してReActループと最小構成のAIエージェントを実装する

---
## 0. 環境準備

In [ ]:
%pip install -q langchain langchain-core langchain-openai

### APIキーの設定

OpenAI APIキーを環境変数へ設定する。キーはコードに直書きせず、実行時に入力する方式とする。

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY を入力する: ")

print("APIキー設定完了" if os.environ.get("OPENAI_API_KEY") else "未設定")

### `openai`のログ設定

今回はLangChainを使用するため実際のAPIコールの内容をログで確認する

In [ ]:
import logging
import os

# OpenAI SDK と HTTP 通信の詳細ログを出力する
os.environ["OPENAI_LOG"] = "debug"

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    force=True,
)

logging.getLogger("openai").setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.DEBUG)
logging.getLogger("httpcore").setLevel(logging.DEBUG)

print("OpenAI DEBUGログを有効化しました")

---
## 1. Tool Use

### 概要

LLM単体は「テキストを生成する」だけ。「作業」はできない。
**Tool Use** は、LLMに「使える道具（関数）」を渡し、必要なときに呼び出させる仕組み。

処理の流れ:

1. ツール定義: LLMに伝えるための関数の名前・説明・引数を定義する
2. LLM呼び出し: ユーザー入力とツール定義をLLMへ渡す
3. LLMの判断: 要求を満たすにはどのツールが必要か判断する
4. 呼び出し生成: LLMがツール名と引数を構造化（JSON）して返す
5. ツール実行: 自分のコードが実際に関数を実行する
6. 観測: 実行結果をLLMへ戻せる形式にする
7. LLM再呼び出し: 観測結果を会話履歴に加えて、もう一度LLMへ渡す
8. 続行: 結果を踏まえて次の手（別ツール or 最終回答）を決める

> LangChainでは`@tool` デコレータの docstring がそのままツールの`description`になる。

In [ ]:
# ツール定義
from langchain_core.tools import tool

@tool(parse_docstring=True)
def get_word_length(word: str) -> int:
    """単語の文字数を返す。長さを数える必要があるときに使う。
    
    Args:
        word: 文字数を数えたい対象の単語。
    """
    return len(word)

# ツールはそれ単体でも呼べる（LLMはまだ関与しない）
print("ツール直接実行:", get_word_length.invoke({"word": "agent"}))
print("名前:", get_word_length.name)
print("説明:", get_word_length.description)

In [ ]:
from langchain_openai import ChatOpenAI

# モデルの設定
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# bind_tools でモデルに「使える道具」を知らせる
model_with_tools = model.bind_tools([get_word_length])

# モデルは答えを直接出さず、「このツールをこの引数で呼べ」という指示(tool_calls)を返す
model_output = model_with_tools.invoke("'agent' という単語は何文字？")
print("本文:", repr(model_output.content))
print("tool_calls:", model_output.tool_calls)

ポイント: モデルは**自分でツールを実行しない**。`tool_calls` という「呼び出し指示」を返すだけ。
実際に実行して結果を戻すのは私たちのコードの役目。これを自動で回すのが次の **ReAct ループ**。

---
## 2. ReAct（Reason + Act）

### 概要

**ReAct** は「考える」と「道具を使う」を交互に繰り返すループ。

```
Thought（思考）   : いま何が必要か考える
Action（行動）    : ツールを呼ぶ
Observation（観測）: 結果を受け取る
   ↑ 必要なだけ繰り返し、十分な情報が揃ったら最終回答
```

Tool Use（1回の呼び出し）を**ループで回す**ことで、複数ステップのタスクを自力で
進める「エージェント」になる。まずはこのループを素の Python で実装して仕組みを理解する。

In [ ]:
import ast, operator, datetime as _dt
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# --- 複数のツールを用意する ---
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def _safe_eval(node):
    if isinstance(node, ast.Expression): return _safe_eval(node.body)
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.BinOp): return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("数式として解釈できない")

@tool(parse_docstring=True)
def calculator(expression: str) -> str:
    """数式を計算して結果を返す。四則演算・べき乗(**)・括弧に対応。

    Args:
        expression: 計算したい数式の文字列。例 '2 * (3 + 4)'。
    """
    try:
        return str(_safe_eval(ast.parse(expression, mode="eval")))
    except Exception as e:
        return f"計算エラー: {e}"

@tool
def current_datetime() -> str:
    """現在のローカル日時をISO形式で返す。"""
    return _dt.datetime.now().isoformat(timespec="seconds")

tools = [calculator, current_datetime]
print("用意したツール:", [t.name for t in tools])

In [ ]:
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

SYSTEM = (
    "あなたは有能なアシスタントです。必要なら calculator / current_datetime を使ってください。"
    "計算は推測せず必ずツールを使い、最後は日本語で簡潔に答えてください。"
)

def react_loop(query: str, max_iter: int = 6) -> str:
    messages = [
        SystemMessage(SYSTEM),
        HumanMessage(query)
    ]
    for step in range(1, max_iter + 1):
        ai = model_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:                       # 観測の必要なし -> 最終回答
            print(f"[step {step}] 最終回答")
            return ai.content
        for call in ai.tool_calls:                  # 行動 -> 観測
            print(f"[step {step}] 行動: {call['name']}({call['args']})")
            obs = str(tools_by_name[call["name"]].invoke(call["args"]))
            print(f"[step {step}] 観測: {obs}")
            messages.append(ToolMessage(content=obs, tool_call_id=call["id"]))
    return "（最大反復回数に達した）"

# 依存のあるタスク: まず今が何年か調べ、その結果を使って築年数を計算する -> 逐次的に複数ステップ回る
print("\n=>", react_loop("2010年に建てられたマンションは、今年で築何年になりますか？"))

---
## 3. 最小構成AIエージェント `minimal-agent`

上のループを再利用できるよう、本体をパッケージへ切り出した。中身は **Tool Use + ReAct の2点のみ**。

```text
src/
└── minimal_agent/
    ├── config.py      # 設定（モデル名・温度・最大反復回数）
    ├── llm.py         # LLM ファクトリ（プロバイダ差し替え点）
    ├── prompts.py     # システムプロンプト（ReAct の振る舞い定義）
    ├── tools/         # ツール（calculator / current_datetime / search_information）
    └── agent.py       # Tool Use + ReAct ループ本体（Agent クラス）
```

In [ ]:
# === Google Colab で実行する場合のみ、次の2行の先頭の # を外して実行 
# !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
# %cd ai-agent-seminar/session04

%pip install -e .

In [ ]:
from minimal_agent import Agent, get_default_tools

print("既定ツール:", [t.name for t in get_default_tools()])

agent = Agent()
answer = agent.run("2010年に建てられたマンションは、今年で築何年になりますか？", verbose=True)
print("\n=== 最終回答 ===")
print(answer)

In [ ]:
# 独自ツールを追加
from langchain_core.tools import tool

@tool
def to_upper(text: str) -> str:
    """英字の文字列を大文字に変換する。
    
    Args:
        text: 大文字に変換したい文字列。
    """
    return text.upper()

custom_agent = Agent(tools=get_default_tools() + [to_upper])
print(custom_agent.run("'hello world' を大文字にして。", verbose=True))